____
### 1. Imports

In [2]:
import gymnasium as gym
import numpy as np
import matplotlib.pyplot as plt
from collections import defaultdict

____
### 2. Exploring Environment

In [3]:
env = gym.make("FrozenLake-v1")

print("Action space:", env.action_space)   # 0(left), 1(down), 2(right), 3(up)
state, _ = env.reset() ## always 0, start at left corner
print("Meaning:")
print("0  1  2  3" \
      "\n4  5  6  7" \
      "\n8  9  10 11" \
      "\n12 13 14 15")

print("Position:", state)
print("\n")


Action space: Discrete(4)
Meaning:
0  1  2  3
4  5  6  7
8  9  10 11
12 13 14 15
Position: 0




In [4]:
def print_env(env):
    position, reward, terminated, truncated, info = env
    print(f"Current Position: {position}")
    print(f"Reward:{reward}")
    print(f"Terminated? {terminated}") 
    print(f"Truncated? {truncated}")
    print(f"Extra Info {info}")   
    print("\n")

In [5]:
# Watch a few random steps
res = 0
print("0  1  2  3" \
      "\n4  5  6  7" \
      "\n8  9  10 11" \
      "\n12 13 14 15")
print("\n")
for i in range(5): # iterating 5 times
    action = env.action_space.sample() #choosing a random action from the sample of 4 possible actions
    next_env = env.step(action) # obtain the new environment after applying the effects of the action on it
    print_env(next_env)

env.close()

0  1  2  3
4  5  6  7
8  9  10 11
12 13 14 15


Current Position: 4
Reward:0
Terminated? False
Truncated? False
Extra Info {'prob': 0.33333333333333337}


Current Position: 4
Reward:0
Terminated? False
Truncated? False
Extra Info {'prob': 0.33333333333333337}


Current Position: 4
Reward:0
Terminated? False
Truncated? False
Extra Info {'prob': 0.33333333333333337}


Current Position: 8
Reward:0
Terminated? False
Truncated? False
Extra Info {'prob': 0.33333333333333337}


Current Position: 8
Reward:0
Terminated? False
Truncated? False
Extra Info {'prob': 0.3333333333333333}




____
### 3. Instantiating the Q-table

In [6]:
# FrozenLake — 16 states, 4 actions, all known upfront
q_table = np.zeros([16, 4])
print("Q table is a numpy array with 16 rows of 4 values each")
print(q_table)
print("each row represents a position and each value corresponds to the action taken at that position")

Q table is a numpy array with 16 rows of 4 values each
[[0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]
 [0. 0. 0. 0.]]
each row represents a position and each value corresponds to the action taken at that position


____
### 4. Hyperparameters

In [7]:
# episodes = number of rounds the simulation will run 
EPISODES = 50_000    

#alpha = amount to update the q-value each time, the learning rate of the model
ALPHA = 0.8       

#gamma = the amount to value future rewards 
GAMMA = 0.95      

#epsilon = exploration rate, eps = 1 means the agent starts at a completely random decision
EPSILON = 1.0

#how much epsilon changes after each episode, kept to small amount so that decay is slow
EPSILON_DECAY = 0.999 # higher decay for fewer episode trainings

____
### 5. Training the model (agent)

In [8]:
def make_decision(eps, q_table, state, env):
    if np.random.random() < eps:
        return env.action_space.sample()
    else:
        return np.argmax(q_table[state])
    

def update_qtable(q_table, state, next_state, action, reward):
    best_nxt_action = np.max(q_table[next_state])
    curr_est = q_table[state][action]
    tgt = reward + GAMMA * best_nxt_action
    q_table[state][action] = curr_est + ALPHA * (tgt - curr_est)

In [9]:
env = gym.make("FrozenLake-v1", is_slippery=True) # creating the environment

rewards_per_episode = []

for episode in range(EPISODES):
    state, i = env.reset()
    total_reward = 0

    while True:
        action = make_decision(EPSILON, q_table, state, env)
        position, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated

        update_qtable(q_table, state, position, action, reward)

        state = position
        total_reward += reward
        if done:
            break
    
    rewards_per_episode.append(total_reward)
    EPSILON *= EPSILON_DECAY
    
    # Print average rewards every 10,000 episodes
    if (episode + 1) % 10000 == 0:
        avg_reward = np.mean(rewards_per_episode[-10000:])
        print(f"Episode {episode + 1}/{EPISODES} - Average Reward (last 10k): {avg_reward:.4f}")

env.close()

print("\nTraining complete!")


Episode 10000/50000 - Average Reward (last 10k): 0.4548
Episode 20000/50000 - Average Reward (last 10k): 0.7072
Episode 30000/50000 - Average Reward (last 10k): 0.6510
Episode 40000/50000 - Average Reward (last 10k): 0.7417
Episode 50000/50000 - Average Reward (last 10k): 0.7351

Training complete!


____
### 6. Evaluating Agent 

In [10]:
def evaluate_agent(q_table, num_rounds):
    env = gym.make("FrozenLake-v1", is_slippery=True)
    successes = 0
    total_steps = []

    for _ in range(num_rounds):
        state, _ = env.reset()
        steps = 0
        while True:
            action = np.argmax(q_table[state])    # pure exploitation, no randomness
            state, reward, terminated, truncated, _ = env.step(action)
            steps += 1

            if terminated or truncated:
                if reward == 1:                   # only +1 means success in FrozenLake
                    successes += 1
                    total_steps.append(steps)
                break

    env.close()

    success_rate = successes / num_rounds
    avg_steps = np.mean(total_steps) if total_steps else 0

    print(f"Results over {num_rounds:,} episodes:")
    print(f"  Success rate: {success_rate*100:.1f}%")
    print(f"  Total successes: {successes:,}")
    print(f"  Avg steps to goal (when successful): {avg_steps:.1f}")

    return success_rate


evaluate_agent(q_table, 100_000)

Results over 100,000 episodes:
  Success rate: 74.1%
  Total successes: 74,112
  Avg steps to goal (when successful): 38.6


0.74112

____
### 7. Creating a Agent as a Class
- implementing all individual functions as methods of the class
- train(), evaluate(), make_decision() and update_qtable() as methods


In [18]:
class FrozenLakeAgent():
    def __init__(this, 
        name : str = "bot",
        alpha : float = 0.8,
        gamma : float = 0.95, 
        epsilon: float = 1.0,
        epsilon_decay : float = 0.999,
        slippery: bool = True
    ):
        this.name = name
        this.alpha = alpha
        this.gamma = gamma
        this.epsilon = epsilon
        this.epsilon_decay = epsilon_decay
        this.q_table = np.zeros([16, 4])
        this.slippery = slippery
        this.env = gym.make("FrozenLake-v1", is_slippery=slippery)

    def make_decision(this, state):
        if np.random.random() < this.epsilon: # at early stage, still exploring
            return this.env.action_space.sample() #exploration 
        else: #at late stage
            return np.argmax(this.q_table[state]) # indexing q-table
    

    def update_qtable(this, state, next_state, action, reward):
        best_nxt_action = np.max(this.q_table[next_state]) #finds the best next action from the table
        curr_est = this.q_table[state][action] #reads the current estimate 
        tgt = reward + this.gamma * best_nxt_action #builds the target value
        this.q_table[state][action] = curr_est + this.alpha * (tgt - curr_est) 
        #moves the old value towards the target by the learning rate
    

    def train(this, train_rds):
        for episode in range(train_rds): #iterate for number of times
            state, _ = this.env.reset()
            total_reward = 0
            while True:
                action = this.make_decision(state)
                next_state, reward, terminated, truncated, _ = this.env.step(action)
                done = terminated or truncated
                this.update_qtable(state, next_state, action, reward)
                state = next_state
                total_reward += reward
                if done:
                    break
            this.epsilon *= this.epsilon_decay #updating epsilon
            '''
            if (episode + 1) % 1000 == 0: #printing every 10,000 episodes
                avg_reward = total_reward / episode
                print(f"[{this.name}] Episode {episode + 1}/{train_rds} - Cumulative Avg Reward: {avg_reward:.4f}")
            '''
        this.env.close()
        # print(f"Training complete for agent {this.name}!")
    
    def evaluate(this, num_rounds=1000): #evaluation 
        env = gym.make("FrozenLake-v1", is_slippery=this.slippery)
        successes = 0
        
        for _ in range(num_rounds):
            state, _ = env.reset()
            while True:
                action = np.argmax(this.q_table[state])  # Pure exploitation
                state, reward, terminated, truncated, _ = env.step(action)
                if terminated or truncated:
                    if reward == 1:
                        successes += 1
                    break
        
        env.close()
        success_rate = successes / num_rounds
        # print(f"[{this.name}] Success rate: {success_rate*100:.1f}% ({successes}/{num_rounds})")
        return success_rate





pipeline to train agent

In [20]:
bot1 = FrozenLakeAgent("bot1")
bot1.train(10000)
success_rate = round(bot1.evaluate() * 100, 2)
print(f"Success Rate: {success_rate} %")

Success Rate: 74.2 %


____
### 8. Fine Tuning hyper parameters to adjust the success rate of the bot


In [ ]:
"name: [alpha, gamma, epsilon, epsilon_decay, episodes to train]"

values = {
    "default" : [],  # baseline
    "lower alpha" : [0.3, 0.95, 1.0, 0.999,  50_000],  
    "medium alpha" : [0.5, 0.95, 1.0, 0.999,  50_000],  
    "lower gamma": [0.8, 0.90, 1.0, 0.999,  50_000],  
    "higher gamma": [0.8, 0.99, 1.0, 0.999,  50_000],
    "faster eps decay" : [0.8, 0.95, 1.0, 0.995,  50_000], 
    "slower eps decay" : [0.8, 0.95, 1.0, 0.9995, 50_000],  
    "default with more episodes" : [0.8, 0.95, 1.0, 0.999, 100_000]
}

collated = []

for name, values in values.items():
    if len(values) == 0:
        alpha, gamma, epsilon, epsilon_decay, episodes = 0.8, 0.95, 1.0, 0.999, 50_000
    else:
        alpha, gamma, epsilon, epsilon_decay, episodes = values
    bot = FrozenLakeAgent(
        name=name,
        alpha=alpha,
        gamma=gamma,
        epsilon=epsilon,
        epsilon_decay=epsilon_decay)
    bot.train(episodes)
    succ = round(bot.evaluate(1000) * 100, 2)
    collated.append((name, succ))
    print(f"Success rate of bot {name} : {succ} %")
    print("---------------------------------------------------------------")

Success rate of bot default : 52.9 %
---------------------------------------------------------------
Success rate of bot lower alpha : 74.1 %
---------------------------------------------------------------
Success rate of bot medium alpha : 73.9 %
---------------------------------------------------------------
Success rate of bot lower gamma : 73.6 %
---------------------------------------------------------------
Success rate of bot higher gamma : 52.6 %
---------------------------------------------------------------
Success rate of bot faster eps decay : 71.9 %
---------------------------------------------------------------
Success rate of bot slower eps decay : 73.1 %
---------------------------------------------------------------
Success rate of bot default with more episodes : 74.0 %
---------------------------------------------------------------


- averaging the models winrates over multiple tries to investigate which hyperparameter adjustment increases winrate the most

In [22]:
def run_n(name, values, n):
    alpha, gamma, epsilon, epsilon_decay, episodes = values
    final = []
    for i in range (0,n):
        bot = FrozenLakeAgent(
            name=name,
            alpha=alpha,
            gamma=gamma,
            epsilon=epsilon,
            epsilon_decay=epsilon_decay)
        bot.train(episodes)
        succ = bot.evaluate(1000) * 100
        final.append(succ)
    avg_succrate = sum(final) / len(final)
    return name, avg_succrate

In [24]:
values = {
    "control" : [0.8, 0.95, 1.0, 0.999, 50_000],  # baseline
    "lower alpha" : [0.3, 0.95, 1.0, 0.999,  50_000],  
    "medium alpha" : [0.5, 0.95, 1.0, 0.999,  50_000],  
    "lower gamma": [0.8, 0.90, 1.0, 0.999,  50_000],  
    "higher gamma": [0.8, 0.99, 1.0, 0.999,  50_000],
    "faster eps decay" : [0.8, 0.95, 1.0, 0.995,  50_000], 
    "slower eps decay" : [0.8, 0.95, 1.0, 0.9995, 50_000],  
    "default with more episodes" : [0.8, 0.95, 1.0, 0.999, 100_000]
}

n = 5

for name, values in values.items():
    name, avg_succrate = run_n(name, values, n)
    dict[name] = round(avg_succrate, 2)
    

- For your own sanity, don't run 15 simulations for each set of parameters, this took more than 2 hours...
- Run a smaller number like 5 for a easier consistency check

In [25]:
for name, succrate in dict.items():
    print(f"{name} bot has average success rate of {succrate}% over {n} rounds of training the same model")
    print("---------------------------------------------------------------")

control bot has average success rate of 73.58% over 5 rounds of training the same model
---------------------------------------------------------------
lower alpha bot has average success rate of 71.22% over 5 rounds of training the same model
---------------------------------------------------------------
medium alpha bot has average success rate of 73.22% over 5 rounds of training the same model
---------------------------------------------------------------
lower gamma bot has average success rate of 64.32% over 5 rounds of training the same model
---------------------------------------------------------------
higher gamma bot has average success rate of 54.06% over 5 rounds of training the same model
---------------------------------------------------------------
faster eps decay bot has average success rate of 51.18% over 5 rounds of training the same model
---------------------------------------------------------------
slower eps decay bot has average success rate of 74.32% over 

Conclusion:
- Slower Epsilon Decay can lead to a higher success rate.
- With a slower epsilon decay, the model has more time to slowly learn and explore while the 'randomness' of it is eroded slowly